# Men's Tournament Results

This notebook is meant to gather tournament finishes from previous years

In [1]:
SEASON = 2025

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneyDetailedResults.csv')

# only using 2012 or later, but looking at previous 4 years as well
df = df.loc[df['Season'] >= 2008, :].reset_index(drop=True)

df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2008,134,1291,69,1164,60,N,0,19,55,5,20,26,32,13,28,10,11,7,3,16,25,57,4,17,6,13,8,26,9,11,2,6,22
1,2008,136,1181,71,1125,70,N,0,25,58,6,21,15,21,13,27,9,15,11,2,17,26,59,8,23,10,15,9,24,12,14,7,2,21
2,2008,136,1242,85,1340,61,N,0,33,61,12,25,7,15,15,23,21,11,10,3,15,21,55,9,25,10,14,13,18,11,16,8,3,18
3,2008,136,1243,80,1425,67,N,0,29,60,7,16,15,26,21,23,15,13,7,1,21,21,50,6,12,19,27,9,18,12,11,6,4,24
4,2008,136,1266,74,1246,66,N,0,23,52,5,13,23,29,15,19,10,7,5,3,18,23,48,8,20,12,17,9,17,13,12,4,5,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057,2024,146,1301,76,1181,64,N,0,28,60,3,13,17,23,8,27,16,4,4,6,16,19,59,5,20,21,26,10,27,11,9,4,5,23
1058,2024,146,1345,72,1397,66,N,0,24,53,3,15,21,33,8,32,16,10,5,2,12,24,62,11,26,7,11,6,17,17,6,8,4,25
1059,2024,152,1163,86,1104,72,N,0,31,62,10,25,14,18,10,25,20,4,4,8,17,26,58,11,23,9,11,7,21,9,7,2,5,15
1060,2024,152,1345,63,1301,50,N,0,22,55,10,25,9,10,10,28,13,14,5,2,8,21,57,5,19,3,4,6,22,10,11,8,3,13


In [3]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] >= 2008, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

df_seeds.insert(1, 'Region Seed', df_seeds['Region'] + df_seeds['Seed'].astype(str).str.zfill(2))

df_seeds

,Season,Region Seed,Seed,Region,Play In,TeamID
0,2008,W01,1,W,False,1314
1,2008,W02,2,W,False,1397
2,2008,W03,3,W,False,1257
3,2008,W04,4,W,False,1450
4,2008,W05,5,W,False,1323
...,...,...,...,...,...,...
1074,2024,Z12,12,Z,False,1241
1075,2024,Z13,13,Z,False,1436
1076,2024,Z14,14,Z,False,1324
1077,2024,Z15,15,Z,False,1443


In [4]:
df_games_won = (
    df
    .groupby(['Season', 'WTeamID'])
    ['WTeamID']
    .count()
    .rename('Games Won')
    .reset_index()
)

df_games_won

,Season,WTeamID,Games Won
0,2008,1116,1
1,2008,1139,1
2,2008,1172,3
3,2008,1181,1
4,2008,1207,1
...,...,...,...
549,2024,1401,1
550,2024,1429,1
551,2024,1447,1
552,2024,1450,1


Remove 1 win from teams who played a play-in game

In [5]:
df_games_won = pd.merge(
    df_games_won,
    df_seeds[['Season', 'TeamID', 'Play In']].rename(columns={'TeamID': 'WTeamID'}),
    how='left',
    on=['Season', 'WTeamID'],
)

df_games_won['Games Won'] -= df_games_won['Play In']

df_games_won

,Season,WTeamID,Games Won,Play In
0,2008,1116,1,False
1,2008,1139,1,False
2,2008,1172,3,False
3,2008,1181,1,False
4,2008,1207,1,False
...,...,...,...,...
549,2024,1401,1,False
550,2024,1429,1,False
551,2024,1447,0,True
552,2024,1450,1,False


In [6]:
df_games_lost = (
    df  # keep play-in games
    .groupby(['Season', 'LTeamID'])
    ['LTeamID']
    .count()
    .rename('Games Lost')
    .reset_index()
)

df_games_lost

,Season,LTeamID,Games Lost
0,2008,1110,1
1,2008,1112,1
2,2008,1116,1
3,2008,1122,1
4,2008,1124,1
...,...,...,...
1057,2024,1443,1
1058,2024,1447,1
1059,2024,1450,1
1060,2024,1458,1


Get results per year for every team

In [7]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\MTeams.csv')

df_teams

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
...,...,...,...,...
375,1476,Stonehill,2023,2025
376,1477,East Texas A&M,2023,2025
377,1478,Le Moyne,2024,2025
378,1479,Mercyhurst,2025,2025


In [8]:
season_teams = [(s, t) for s in range(2008, SEASON) for t in df_teams['TeamID'].unique()]

len(season_teams)

6460

In [9]:
from tqdm.autonotebook import tqdm

season_team_results = []

for season, team in tqdm(season_teams):
    if df_games_won.loc[(df_games_won['Season'] == season) & (df_games_won['WTeamID'] == team), :].shape[0] != 0:
        # team won at least one tournament game
        season_team_results.append((
            season, 
            team, 
            df_games_won.loc[(df_games_won['Season'] == season) & (df_games_won['WTeamID'] == team), 'Games Won'].iloc[0]
        ))
    elif df_games_lost.loc[(df_games_lost['Season'] == season) & (df_games_lost['LTeamID'] == team), 'Games Lost'].shape[0] != 0:
        # team did not win non-play-in game
        season_team_results.append((season, team, 0))
    else:
        # team did not make tournament
        season_team_results.append((season, team, -1))

len(season_team_results)

C:\Users\mhugh\AppData\Local\Temp\ipykernel_23396\2733837186.py:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/6460 [00:00<?, ?it/s]

6460

In [10]:
df_tournament_results = pd.DataFrame(
    season_team_results,
    columns=['Season', 'TeamID', 'Result'],
)

id_to_name = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_tournament_results.insert(
    df_tournament_results.columns.get_loc('TeamID') + 1, 
    'Team',
    df_tournament_results['TeamID'].map(id_to_name)
)

df_tournament_results

,Season,TeamID,Team,Result
0,2008,1101,Abilene Chr,-1
1,2008,1102,Air Force,-1
2,2008,1103,Akron,-1
3,2008,1104,Alabama,-1
4,2008,1105,Alabama A&M,-1
...,...,...,...,...
6455,2024,1476,Stonehill,-1
6456,2024,1477,East Texas A&M,-1
6457,2024,1478,Le Moyne,-1
6458,2024,1479,Mercyhurst,-1


Set 2020 to NaN due to tournament cancellation

In [11]:
import numpy as np

df_tournament_results.loc[df_tournament_results['Season'] == 2020, 'Result'] = np.nan

In [12]:
df_tournament_results['Past 4 Years Tournament Results'] = (
    df_tournament_results
    .groupby(['TeamID'])
    ['Result']
    .rolling(window=4, min_periods=1)
    .mean()
    .reset_index()
    .set_index('level_1')
)['Result']

df_tournament_results.rename(
    columns={
    'Result': 'Past Year Tournament Result'
    }, 
    inplace=True
)

df_tournament_results['Season'] += 1  # shift by a year so results are from past instead of the current tourney

df_tournament_results = df_tournament_results.loc[df_tournament_results['Season'] >= 2012, :].reset_index(drop=True)

df_tournament_results

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2012,1101,Abilene Chr,-1.0,-1.0
1,2012,1102,Air Force,-1.0,-1.0
2,2012,1103,Akron,0.0,-0.5
3,2012,1104,Alabama,-1.0,-1.0
4,2012,1105,Alabama A&M,-1.0,-1.0
...,...,...,...,...,...
5315,2025,1476,Stonehill,-1.0,-1.0
5316,2025,1477,East Texas A&M,-1.0,-1.0
5317,2025,1478,Le Moyne,-1.0,-1.0
5318,2025,1479,Mercyhurst,-1.0,-1.0


In [14]:
df_tournament_results.to_csv(
    '../data/preprocessed/kaggle/tournament_results.csv', 
    index=False,
)

'Done'

'Done'